# 02a · From PyTorch to SpiNNaker2 via NIR

Take a small spiking network defined in snnTorch (a PyTorch library), export it to NIR, convert it into a py-spinnaker2 network, run it, and check spike for spike that it computes the same as the original. Runs on Brian2; cluster participants run it on the chip in section 6.

Adapted from the vendor's [tutorial 08, "The NIR workflow"](https://gitlab.com/spinnaker2/py-spinnaker2/-/blob/v0.8.2/tutorials/deep_dive/08_nir.ipynb) (SpiNNcloud, Apache-2.0), section 3. Each section ends with **Go deeper**: where to read more. Collapsed blocks (▶) hold background for anyone who wants more; the exercises don't need them.

**Workshop notes:** the folder [`01_knowledge_base`](../01_knowledge_base/) collects background written while preparing this workshop (spiking networks, the hardware, energy), with figures and further links. It is a reading collection for anyone who wants to delve deeper, not a primary source.

## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import snntorch as snn
import torch
import torch.nn as nn
from snntorch import utils
from snntorch.export_nir import export_to_nir
from spinnaker2 import brian2_sim, helpers, s2_nir

USE_HARDWARE = False


def backend():
    """A fresh backend for every run: Brian2, or one 48-chip SpiNNaker2 board."""
    if USE_HARDWARE:
        from spinnaker2 import hardware

        return hardware.SpiNNcloud48NodeBoard()
    return brian2_sim.Brian2Backend()

## 1. A spiking network in snnTorch

Four inputs, a hidden layer of 5 LIF neurons, an output layer of 2 LIF neurons. In snnTorch, `nn.Linear` holds the weights and `snn.Leaky` is a layer of LIF neurons with decay factor `beta`, the same as `alpha_decay` in hands-on 1.

The network is not trained: the weights are random (`manual_seed` makes them reproducible). Training follows in notebook 02b; here the conversion is the point.

<details><summary>The update rule of <code>snn.Leaky</code></summary>

Per time step, `snn.Leaky` computes $U[t+1] = \beta\, U[t] + I_\text{in}[t+1]$, where $I_\text{in}$ is the output of the preceding `nn.Linear`. The neuron spikes when $U$ exceeds the threshold (1 by default). With `reset_mechanism="zero"`, $U$ is set to 0 after a spike; with `"subtract"` (the default), the threshold is subtracted. This is the same update as the `lif` neuron of hands-on 1, with `beta` in the role of `alpha_decay`. `init_hidden=True` lets the layer keep its membrane potential internally, so `nn.Sequential` works. Source: the `snn.Leaky` docstring and [snnTorch tutorial 2](https://snntorch.readthedocs.io/en/latest/tutorials/tutorial_2.html).

</details>

**Go deeper:** building and simulating networks in snnTorch: [snnTorch tutorials](https://snntorch.readthedocs.io/en/latest/tutorials/index.html), 1–3; the library and its design: [Eshraghian et al. 2023, Proc. IEEE](https://doi.org/10.1109/JPROC.2023.3308088). Workshop notes: [SNNs and deep learning](../01_knowledge_base/01_primer_snn_basics.md#8-snns-and-deep-learning).

In [ ]:
torch.manual_seed(9)
snn_net = nn.Sequential(
    nn.Linear(4, 5, bias=False),  # 4 inputs -> 5 hidden neurons
    snn.Leaky(beta=0.9, init_hidden=True, reset_mechanism="zero"),  # 5 LIF neurons, reset to 0 after a spike
    nn.Linear(5, 2, bias=False),  # 5 hidden -> 2 output neurons
    snn.Leaky(beta=0.9, init_hidden=True, reset_mechanism="zero", output=True),  # 2 LIF neurons
)
print(snn_net)

## 2. Export to NIR

NIR (Neuromorphic Intermediate Representation) describes the network as a graph of building blocks: weight layers (`Linear`) and neuron groups (`LIF`), connected by edges. snnTorch runs the model once on an example input to record this graph; only the size of the input matters, not its values.

<details><summary>What NIR is, and why it exists</summary>

Every spiking framework and chip describes neurons in its own way, so a trained network used to be tied to one of them. NIR is a shared file format, comparable to ONNX for deep learning: each building block (a "primitive") is defined by a continuous-time equation, independent of any simulator's time step. The LIF primitive, for example, is

$$\tau\, \dot v = (v_\text{leak} - v) + R\, I, \qquad \text{spike if } v > v_\text{thr}.$$

Each framework exports to it, each chip imports from it and discretises the equations with its own step. [Pedersen et al. 2024 (Nature Communications)](https://doi.org/10.1038/s41467-024-52259-9) introduce NIR and show it with 7 simulators and 4 hardware platforms, SpiNNaker2 among them. The list of primitives: [NIR documentation](https://neuroir.org/docs/primitives/).

</details>

In [ ]:
nir_graph = export_to_nir(snn_net, torch.zeros(4))
nir_graph.infer_types()  # fill in the input and output shapes of every node
s2_nir.model_summary(nir_graph)

## 3. Conversion settings

`s2_nir.from_nir` turns the NIR graph into `snn.Population` and `snn.Projection` objects, the same objects as in hands-on 1. A `ConversionConfig` controls the translation. **Rule for models from snnTorch:** `dt = 1e-4`, integrator `FORWARD`, and the reset rule of the snnTorch model. Otherwise the chip runs a slightly different network, without any error message:

| Setting | Value for snnTorch | Why |
|---|---|---|
| `dt` | `1e-4` | snnTorch stores `beta` as a time constant $\tau = dt / (1 - \beta)$ with a fixed $dt = 10^{-4}$; only the same `dt` gives back `beta` |
| `integrator` | `FORWARD` | snnTorch computes $v[t+1] = \beta\, v[t] + \text{input}$, a forward Euler step. The py-spinnaker2 default `EXPONENTIAL` would give a decay of 0.905 instead of 0.9 |
| `reset` | `ZERO` here | NIR does not store the reset rule, so it has to be set by hand to match `reset_mechanism` |

A fourth difference concerns the comparison: the chip delays every spike by one step per projection, snnTorch does not (section 5).

<details><summary>Forward and exponential, in numbers</summary>

snnTorch exports `beta` as a time constant $\tau = \Delta t / (1 - \beta)$ with $\Delta t = 10^{-4}$ s; for $\beta = 0.9$ that is $\tau = 10^{-3}$ s. py-spinnaker2 turns $\tau$ back into a decay factor per step:

- `FORWARD` ([forward Euler](https://en.wikipedia.org/wiki/Euler_method)): $\alpha = 1 - \Delta t / \tau = 0.9$, exactly snnTorch's `beta`
- `EXPONENTIAL` (exact solution of the decay): $\alpha = e^{-\Delta t / \tau} = e^{-0.1} \approx 0.905$

Both are valid ways to discretise the NIR equation; only the forward one reproduces what snnTorch computed. A different `dt` changes $\alpha$ as well: with `dt = 1e-3`, forward Euler gives $\alpha = 0$, a neuron without memory. Source: `convert_LIF` in py-spinnaker2's `s2_nir.py`.

</details>

<details><summary>What <code>scale_weights</code> does</summary>

NIR layers become `lif_no_delay` populations on the chip, with 8-bit integer weights (−127…127) and no synaptic delays (hence `conn_delay = 0`). `scale_weights = True` multiplies the weights of each layer so that the largest one becomes 127, and multiplies the thresholds by the same factor, so the network behaves the same apart from rounding. With `weight_scale_percentile` below 100, a percentile of the weights is mapped to 127 instead, which helps when a few outliers would otherwise squeeze all other weights into a few integer values. Source: [py-spinnaker2 API docs, s2_nir](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/s2_nir-api.html).

</details>

In [ ]:
def make_config(integrator=s2_nir.IntegratorMethod.FORWARD, reset=s2_nir.ResetMethod.ZERO):
    cfg = s2_nir.ConversionConfig()
    cfg.dt = 1e-4
    cfg.integrator = integrator
    cfg.reset = reset
    cfg.output_record = ["spikes"]
    cfg.conn_delay = 0
    cfg.scale_weights = True  # scale float weights into the chip's integer range
    return cfg


net, inp, outp = s2_nir.from_nir(nir_graph, make_config())
for pop in net.populations:
    print(f"population {pop.name}: {pop.size} neurons, model {pop.app_class.app_name}")
print(f"{len(net.projections)} projections")

## 4. Input and run

The input population takes spike times, as in hands-on 1. Here each input neuron fires at random in 40 % of the steps. The hidden layer is recorded too, so all layers can be compared.

In [ ]:
steps = 50
rng = np.random.default_rng(seed=42)
input_data = (rng.random((4, steps)) < 0.4).astype(int)  # 1 = spike, one row per input neuron
input_spikes = {i: list(np.nonzero(input_data[i])[0]) for i in range(4)}


def convert_and_run(cfg):
    """Convert the NIR graph with `cfg`, run it, return hidden and output spikes."""
    net, inp, outp = s2_nir.from_nir(nir_graph, cfg)
    inp[0].params = input_spikes
    hidden = [p for p in net.populations if p not in inp and p not in outp][0]
    hidden.record = ["spikes"]
    backend().run(net, steps + 2)  # 2 extra steps for the delays of the two projections
    return hidden.get_spikes(), outp[0].get_spikes()


s2_hidden, s2_output = convert_and_run(make_config())

fig, axes = plt.subplots(3, 1, figsize=(9, 5), sharex=True)
for ax, (spikes, n, name) in zip(axes, [(input_spikes, 4, "input"), (s2_hidden, 5, "hidden"), (s2_output, 2, "output")]):
    idx, t = helpers.spike_times_dict_to_arrays(spikes)
    ax.plot(t, idx, "|", ms=10, mew=1.5)
    ax.set(ylabel=f"{name}\nneuron", ylim=(-0.5, n - 0.5), yticks=range(n))
axes[-1].set_xlabel("time step")
fig.suptitle("the snnTorch network, converted via NIR, running in py-spinnaker2")
plt.tight_layout()
plt.show()

## 5. Same network? Compare with snnTorch

Run the original snnTorch model on the same input. Before comparing, shift the py-spinnaker2 spikes back by one step per projection: 1 for the hidden layer, 2 for the output layer.

<details><summary>Why one step per projection</summary>

In snnTorch, a spike computed in step $t$ reaches the next layer in the same step $t$, because the whole forward pass runs within one step. On the chip, each layer runs on its own cores and a spike travels to the next core as a packet, which is processed in the following step (the `delay + 1` rule of hands-on 1, with `delay = 0`). A network with $L$ projections therefore produces its output $L$ steps later. For classification by spike count, this hardly matters; for spike-for-spike comparisons and for networks that react in real time, it does.

</details>

In [ ]:
def run_snntorch():
    """Hidden and output spike times of the original snnTorch model."""
    utils.reset(snn_net)  # clear the membrane state left over from the export
    lin1, lif1, lin2, lif2 = snn_net
    hidden, output = {i: [] for i in range(5)}, {i: [] for i in range(2)}
    for t in range(steps):
        x = torch.tensor(input_data[:, t], dtype=torch.float32)
        spk1 = lif1(lin1(x))
        spk2, _ = lif2(lin2(spk1))
        for i in np.nonzero(spk1.detach().numpy())[0]:
            hidden[int(i)].append(t)
        for i in np.nonzero(spk2.detach().numpy())[0]:
            output[int(i)].append(t)
    return hidden, output


def shift(spikes, d, n):
    """Shift spikes d steps earlier, undoing the chip's delay of one step per projection."""
    return {i: [int(t) - d for t in spikes.get(i, []) if d <= t < steps + d] for i in range(n)}


def compare(s2_hidden, s2_output, label):
    ref_hidden, ref_output = run_snntorch()
    ok_hidden = shift(s2_hidden, 1, 5) == ref_hidden
    ok_output = shift(s2_output, 2, 2) == ref_output
    n_ref = sum(len(v) for v in ref_hidden.values()) + sum(len(v) for v in ref_output.values())
    print(f"{label}: hidden layer {'identical' if ok_hidden else 'DIFFERENT'}, output layer {'identical' if ok_output else 'DIFFERENT'} ({n_ref} snnTorch spikes)")


compare(s2_hidden, s2_output, "correct settings")

**Try:** break the conversion on purpose and watch the comparison fail. Run each line, one at a time.

1. Use the py-spinnaker2 default integrator.
2. Use the wrong reset rule.
3. Compare without shifting the spikes: replace `shift(s2_hidden, 1, 5)` by `shift(s2_hidden, 0, 5)` in `compare`, or simply reason about what would happen.

None of these mistakes raises an error. That is why the comparison with the original model is worth doing for every converted network.

**Discuss:** which of these differences would you notice in a classifier that counts output spikes, and which only in a spike-for-spike comparison?

**Go deeper:** tutorial 08, section 2 lists the NIR primitives py-spinnaker2 supports and how they map to its neuron models; section 3.6 does the same sanity check; section 4 names the settings that matter for large trained models (weight percentile, neurons per core) and links two trained examples: the event-camera digits of the instructor demo and the end-to-end example of notebook 02b. Conversion settings: [py-spinnaker2 user guide, NIR](https://spinnaker2.gitlab.io/py-spinnaker2/user_guide/qs-nir.html). Workshop notes: the [conversion rule for snnTorch models](../01_knowledge_base/01_primer_snn_basics.md#8-snns-and-deep-learning), with a reading list on NIR.

In [ ]:
compare(*convert_and_run(make_config(integrator=s2_nir.IntegratorMethod.EXPONENTIAL)), "EXPONENTIAL integrator")

In [ ]:
compare(*convert_and_run(make_config(reset=s2_nir.ResetMethod.SUBTRACT)), "SUBTRACT reset")

## 6. Run on the SpiNNaker2 chip (cluster participants)

Set `USE_HARDWARE = True` and run the cell: the same conversion runs on one 48-chip board (about one minute) and is compared with snnTorch.

In [ ]:
USE_HARDWARE = False  # set to True on the cluster

compare(*convert_and_run(make_config()), "SpiNNaker2 chip" if USE_HARDWARE else "Brian2 (USE_HARDWARE is False)")